<a href="https://colab.research.google.com/github/kirilpoukalov/BindCraft2/blob/main/bindcraft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BindCraft 2

<img src="https://raw.githubusercontent.com/kirilpoukalov/BindCraft2/refs/heads/main/docs/bc2_header.png" width="100%">

**Design protein binders around the biology of your experiment.**

BindCraft 2 brings de novo miniproteins, scaffolded binders, cyclic peptides and multistate
design into one workflow. Describe your target, choose the kind of binder you want, and add the
properties that matter for your experiment.

It optimises a sequence through AlphaFold 2, redesigns it with ProteinMPNN, then re-predicts
each candidate with models that did not shape it and keeps the ones that pass its filters. You
get sequences, predicted complexes and ranked results, with measurements of the interface, fold
and molecular properties to choose from.


Before you start:

- **Runtime → Change runtime type → T4 GPU.** BindCraft 2 does not run on CPU.
- Keep the target small. A 16 GB T4 fits about 250 residues of target plus binder, and cell 5
  works yours out before you start.
- Many attempts go into one accepted design. To watch the whole notebook work in a few minutes
  instead, tick `no_filters` in the run cell: it accepts the first attempt, and means nothing.
- The AlphaFold weights are 5.3 GB. Cell 1 can keep them on your Drive so later sessions start
  in seconds.


[This fork on GitHub](https://github.com/kirilpoukalov/BindCraft2) ·
[Upstream BindCraft 2](https://github.com/PacesaLab/BindCraft2) ·
[Settings](https://github.com/kirilpoukalov/BindCraft2/blob/main/docs/reference.md) ·
[Outputs](https://github.com/kirilpoukalov/BindCraft2/blob/main/docs/outputs.md)

## Colab Troubleshooting

| Problem | Fix |
| --- | --- |
| No GPU | Runtime → Change runtime type → T4 GPU. |
| `campaign refused:` | Your combination of settings is currently not supported. Every reason is printed. Change settings in cells 3–4, re-run cell 5, then cell 6. |
| Out of GPU memory | trim the target or shorten the binder. |
| Session disconnected | Run cell 6 again. It carries on if results are on Drive. |


In [ ]:
#@title 1 · Set up { display-mode: "form" }
#@markdown Installs BindCraft 2 and the AlphaFold weights. A few minutes the first time.
#@markdown
#@markdown Drive holds your results, so a campaign cut off by a Colab timeout can carry on.
#@markdown `keep_weights_on_drive` is the trade worth making on a slow session: off, the 5.3 GB
#@markdown of weights come down again every session; on, they sit on your Drive and every
#@markdown session after the first starts in seconds. The compiled-graph cache follows them.
save_results_to_drive = True  #@param {type:"boolean"}
keep_weights_on_drive = True  #@param {type:"boolean"}
drive_folder = "BC2"  #@param {type:"string"}
accelerator = "auto"  #@param ["auto", "cuda13", "cuda12"]

import json, os, re, shutil, subprocess, sys, time
from pathlib import Path

RUNTIME = Path("/content") if os.access("/content", os.W_OK) else Path.home()

BC2_HOME = RUNTIME / "bc2_work"
ON_DRIVE = False
if save_results_to_drive or keep_weights_on_drive:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BC2_HOME = Path("/content/drive/MyDrive") / drive_folder
        ON_DRIVE = True
    except Exception as problem:
        print(f"Drive did not mount: {problem}")
        print("Allow third-party cookies for colab.research.google.com, or mount it from the")
        print("folder icon on the left. Results go on the runtime for now.")

# BC2 reads both variables at startup, so they are set before it is ever imported.
CACHE_DIR = (BC2_HOME / "cache") if (keep_weights_on_drive and ON_DRIVE) else (RUNTIME / "bindcraft_cache")
RESULTS_ROOT = BC2_HOME / "results"
TARGET_DIR = BC2_HOME / "targets"
for directory in (CACHE_DIR, RESULTS_ROOT, TARGET_DIR):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["BINDCRAFT_WEIGHTS"] = str(CACHE_DIR)
os.environ["JAX_COMPILATION_CACHE_DIR"] = str(CACHE_DIR / "compile_cache")

print(f"results:   {RESULTS_ROOT}")
print(f"weights:   {CACHE_DIR}"
      + ("  (on Drive, so they are here next session)" if keep_weights_on_drive and ON_DRIVE
         else "  (this runtime, fetched again each session)"))
print(f"free disk: {shutil.disk_usage(RUNTIME).free / 1e9:.0f} GB (the weights need 11 GB while unpacking)")
started = time.time()
REPO = RUNTIME / "BC2"
GITHUB_REPOSITORY = "https://github.com/kirilpoukalov/BindCraft2.git"

def card_facts():
    """Some drivers do not answer every field. A half-described card is still a card."""
    for fields in ("name,memory.total,compute_cap", "name,memory.total", "name"):
        probe = subprocess.run(["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader"],
                               capture_output=True, text=True)
        if probe.returncode == 0 and probe.stdout.strip():
            answered = [field.strip() for field in probe.stdout.splitlines()[0].split(",")]
            return answered + [""] * (3 - len(answered))
    return None

facts = card_facts()
if facts is None:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again.")
GPU_NAME, gpu_memory, GPU_COMPUTE = facts
try:
    GPU_MEMORY_GB = float(gpu_memory.split()[0]) / 1024
except (IndexError, ValueError):
    GPU_MEMORY_GB = 0.0
print(f"GPU: {GPU_NAME}" + (f", {GPU_MEMORY_GB:.0f} GB" if GPU_MEMORY_GB else ""))
if GPU_COMPUTE and float(GPU_COMPUTE) < 8.0:
    print("No native bfloat16 on this card, so designing is slower than on an L4 or A100.")

def git(*arguments):
    """Run git and show what it said if it failed. No terminal here, so anything git wants to
    ask has to fail rather than wait for an answer nobody can type."""
    done = subprocess.run(["git", *[str(argument) for argument in arguments]],
                          capture_output=True, text=True,
                          env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if done.returncode != 0:
        print((done.stdout + done.stderr).strip())
    return done

BRANCH = "main"

def fetch_source():
    """Clone once, then move the checkout to the head of main."""
    if not (REPO / ".git").exists():
        shutil.rmtree(REPO, ignore_errors=True)
        if git("clone", "--quiet", GITHUB_REPOSITORY, REPO).returncode != 0:
            raise SystemExit(f"Could not clone {GITHUB_REPOSITORY}. Check the runtime's network.")
    if git("-C", REPO, "fetch", "--quiet", "origin", BRANCH).returncode == 0:
        git("-C", REPO, "checkout", "--quiet", "FETCH_HEAD")
    return BRANCH

SOURCE = fetch_source()
revision = subprocess.run(["git", "-C", str(REPO), "describe", "--always"],
                          capture_output=True, text=True).stdout.strip()
print(f"source: {SOURCE}" + (f" ({revision})" if revision else ""))

def stream(command, quiet_pattern=None, **extra):
    """Run a command and print it as it goes. Lines matching quiet_pattern are kept but not
    printed, so pip and a 5.3 GB download do not fill the cell."""
    process = subprocess.Popen([str(part) for part in command], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, **extra)
    kept = []
    for line in process.stdout:
        kept.append(line)
        if quiet_pattern is None or not re.search(quiet_pattern, line):
            print(line, end="")
    return process.wait(), "".join(kept)

def driver_cuda_major():
    listing = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout               if shutil.which("nvidia-smi") else ""
    found = re.search(r"CUDA Version:\s*(\d+)", listing)
    return int(found.group(1)) if found else 0

def chosen_accelerator():
    """install.sh's rule, applied here: the driver's CUDA major version picks the wheels, and a
    card below compute capability 7.5 takes CUDA 12 however new the driver is."""
    if accelerator != "auto":
        return accelerator
    picked = "cuda12" if driver_cuda_major() == 12 else "cuda13"
    if picked == "cuda13" and GPU_COMPUTE and float(GPU_COMPUTE) < 7.5:
        return "cuda12"
    return picked

if sys.version_info < (3, 12):
    raise SystemExit(f"This runtime is Python {sys.version.split()[0]} and BC2 needs 3.12 or "
                     "newer. Pick a newer Colab runtime.")

ACCELERATOR = chosen_accelerator()
PYTHON = sys.executable
LAUNCHER = [PYTHON, "bindcraft.py"]

# Editable, because settings/ and scaffolds/ sit beside the package and are found relative to
# it. PIP_BREAK_SYSTEM_PACKAGES is what the shipped container sets for a system interpreter.
PIP_ENVIRONMENT = {**os.environ, "PIP_BREAK_SYSTEM_PACKAGES": "1",
                   "PIP_DISABLE_PIP_VERSION_CHECK": "1"}
PIP_NOISE = r"already satisfied|^\s*(Collecting|Downloading|Using cached|Installing collected|Successfully|Preparing|Building|Created wheel|Stored in|Attempting uninstall|Found existing|Uninstalling|Obtaining|Checking|Getting|Installing build|Requirement|WARNING|ERROR: pip|\s*\|)"

print(f"installing for {ACCELERATOR}")
stream([PYTHON, "-m", "pip", "install", "-e", f".[{ACCELERATOR}]"],
       quiet_pattern=PIP_NOISE, cwd=REPO, env=PIP_ENVIRONMENT)

def package_is_sound():
    """selfcheck names every module the accelerator needs and the checkpoints inside the
    package. --shipped-only leaves the AlphaFold weights out of it, which come next."""
    done = subprocess.run([PYTHON, "-m", "bindcraft.selfcheck", ACCELERATOR, "--shipped-only"],
                          cwd=REPO, capture_output=True, text=True)
    return done.returncode == 0, (done.stdout + done.stderr).strip()

sound, complaint = package_is_sound()
if not sound:
    # The CUDA extra also pulls cuEquivariance, whose ops wheel is the piece most likely to have
    # no build for a given runtime. Those kernels are optional -- use_cueq is off unless a
    # campaign asks -- so the package with plain jax is a complete install without them.
    print(f"{complaint}\nretrying without the optional cuEquivariance kernels")
    subprocess.run([PYTHON, "-m", "pip", "install", "--quiet", "-e", "."],
                   cwd=REPO, env=PIP_ENVIRONMENT, check=False)
    subprocess.run([PYTHON, "-m", "pip", "install", "--quiet",
                    f"jax[{ACCELERATOR}]>=0.11,<0.12", "cuequivariance-jax>=0.11,<0.12"],
                   cwd=REPO, env=PIP_ENVIRONMENT, check=False)
    sound, complaint = package_is_sound()
if not sound:
    raise SystemExit(f"BC2 is not installed. Missing:\n{complaint}\n\n"
                     f"This is Python {sys.version.split()[0]} with {ACCELERATOR}. Try "
                     f"accelerator = {'cuda12' if ACCELERATOR != 'cuda12' else 'cuda13'}.")
print(f"package: ready ({ACCELERATOR})")

def bindcraft(*arguments, quiet=False):
    """Run a bindcraft command through the launcher, which finds whichever environment the
    package went into."""
    done = subprocess.run([*LAUNCHER, *[str(argument) for argument in arguments]],
                          cwd=REPO, env=os.environ, capture_output=True, text=True)
    if not quiet:
        print((done.stdout + done.stderr).rstrip())
    return done

SEQUENCE_SUFFIXES = (".fasta", ".fa", ".faa")

def chain_residues(path):
    """Residues per chain, or per FASTA record, read with the biotite BC2 installed."""
    script = """
import json, sys
from pathlib import Path
path = Path(sys.argv[1])
if path.suffix.lower() in (".fasta", ".fa", ".faa"):
    from biotite.sequence.io.fasta import FastaFile, get_sequences
    counts = {name: len(sequence) for name, sequence in get_sequences(FastaFile.read(str(path))).items()}
else:
    from biotite.structure.io import load_structure
    structure = load_structure(str(path), model=1)
    alpha = structure[structure.atom_name == "CA"]
    residues = {}
    for chain, residue in zip(alpha.chain_id, alpha.res_id):
        residues.setdefault(str(chain), set()).add(int(residue))
    counts = {chain: len(numbers) for chain, numbers in residues.items()}
print(json.dumps(counts))
"""
    read = subprocess.run([PYTHON, "-c", script, str(path)], cwd=REPO, capture_output=True, text=True)
    return json.loads(read.stdout) if read.returncode == 0 and read.stdout.strip() else {}

def parameter_directory():
    """Where the weights are, if they are here. BC2 accepts either layout."""
    return next((directory for directory in (CACHE_DIR / "alphafold", CACHE_DIR / "alphafold" / "params")
                 if any(directory.glob("params_*.npz"))), None)

# One download. The transfer prints a progress line per 4 MB, so those are dropped.
if parameter_directory() is None:
    print("weights: downloading 5.3 GB, a few minutes")
    weights_status, _ = stream([*LAUNCHER, "fetch-weights"], quiet_pattern=r"of 5\.3 GB",
                               cwd=REPO, env=os.environ)
    if weights_status != 0:
        raise SystemExit("The weights did not arrive. Run this cell again; an interrupted "
                         "download is re-fetched.\nIf it ran out of space, delete the runtime "
                         "and start a fresh one.")

# Pin the directory they are in. BINDCRAFT_AF2_PARAMS overrides every other lookup, so no later
# cell can decide they are missing and download them a second time.
PARAMETERS = parameter_directory()
if PARAMETERS is None:
    raise SystemExit(f"No params_*.npz under {CACHE_DIR / 'alphafold'}. Run this cell again.")
os.environ["BINDCRAFT_AF2_PARAMS"] = str(PARAMETERS)
print(f"weights: ready, {len(list(PARAMETERS.glob('params_*.npz')))} checkpoints")

backend = subprocess.run([PYTHON, "-c", "import jax; print(jax.default_backend())"],
                         cwd=REPO, capture_output=True, text=True).stdout.strip()
print(f"jax: {backend or 'did not start'}")
if backend != "gpu":
    print("JAX did not take the GPU. Restart the runtime and run this cell again.")
print(f"ready in {time.time() - started:.0f} s")

---

<a name="load-a-campaign"></a>
## Load a campaign you ran before

Skip this to design something new.

In [ ]:
#@title 2 · Campaign { display-mode: "form" }
#@markdown Pick a campaign you ran before, or name a new one. A new campaign is saved when
#@markdown you run it.
try:
    import ipywidgets as widgets
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "ipywidgets"], check=True)
    import ipywidgets as widgets
from IPython.display import display

# Some Colab runtimes need this before a widget will draw.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

NEW = "a new campaign"

def previous_campaigns():
    """Every folder here that a campaign wrote, newest first."""
    found = {path.parent for path in RESULTS_ROOT.glob("*/campaign.json")}
    found |= {path.parent.parent for path in RESULTS_ROOT.glob("*/3_Ranked/!_Ranked.csv")}
    return sorted(found, key=lambda folder: folder.stat().st_mtime, reverse=True)

picker = widgets.Dropdown(options=[NEW] + [folder.name for folder in previous_campaigns()],
                          value=NEW, description="campaign",
                          style={"description_width": "160px"},
                          layout=widgets.Layout(width="440px"))
new_name = widgets.Text(description="name it", placeholder="left empty: named after the target",
                        style={"description_width": "160px"},
                        layout=widgets.Layout(width="440px"))
elsewhere = widgets.Text(description="or from a different a path", placeholder="a campaign copied from elsewhere",
                         style={"description_width": "160px"},
                         layout=widgets.Layout(width="640px"))
chosen = widgets.Output()

def campaign_folder():
    """The campaign these cells work on: the one picked here, the one named here, else whatever
    the run cell last saved."""
    if elsewhere.value.strip():
        return Path(elsewhere.value.strip())
    if picker.value != NEW:
        return RESULTS_ROOT / picker.value
    if new_name.value.strip():
        return RESULTS_ROOT / new_name.value.strip()
    return globals().get("PROJECT_FOLDER")

def choose(change=None):
    global CAMPAIGN_FILE, PROJECT_FOLDER, REUSED_CAMPAIGN
    with chosen:
        chosen.clear_output(wait=True)
        folder = campaign_folder()
        REUSED_CAMPAIGN = picker.value != NEW or bool(elsewhere.value.strip())
        new_name.layout.display = None if picker.value == NEW else "none"
        if not REUSED_CAMPAIGN:
            named = new_name.value.strip()
            print(f"A new campaign, saved to {RESULTS_ROOT / named}" if named else
                  "A new campaign, named after its target unless you name it here.")
            return
        if folder is None or not folder.exists():
            print(f"No folder at {folder}")
            return
        PROJECT_FOLDER = folder
        CAMPAIGN_FILE = folder / "campaign.json"
        print(folder)
        if CAMPAIGN_FILE.exists():
            settings = json.loads(CAMPAIGN_FILE.read_text())
            for key in ("target", "modality", "binder_lengths", "number_of_final_designs",
                        "max_trajectories"):
                if key in settings:
                    print(f"  {key}: {settings[key]}")
            if "targets" in settings:
                print(f"  target: {settings['targets'][0].get('name')}")
        else:
            print("  no campaign.json here, so it cannot be re-run; Results still read it")
        accepted = folder / "3_Ranked" / "!_Ranked.csv"
        print(f"  accepted so far: {sum(1 for _ in accepted.open()) - 1}" if accepted.exists()
              else "  nothing accepted yet")
        print("Cell 6 carries it on. Cell 7 reads it.")

picker.observe(choose, names="value")
new_name.observe(choose, names="value")
elsewhere.observe(choose, names="value")
display(widgets.VBox([picker, new_name, elsewhere, chosen]))
choose()

---

## Setup a design campaign


In [ ]:
#@title 3 · Targets { display-mode: "form" }
#@markdown Add a target, then edit it below. Select the residues to bind
#@markdown and the residues to keep clear, and whether the binder should bind it or not.

try:
    import ipywidgets as widgets
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "ipywidgets"], check=True)
    import ipywidgets as widgets
from IPython.display import display

# Some Colab runtimes need this before a widget will draw.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

if "REPO" not in globals():
    raise SystemExit("Run cells 1 and 2 first.")

SEQUENCE_SUFFIXES = (".fasta", ".fa", ".faa")
EXAMPLES = sorted(path.stem for path in (REPO / "settings" / "target").glob("*.json"))
ROLES = ("bind it", "avoid binding it (off-target)")
STYLE = {"description_width": "170px"}

def wide():
    """A Layout belongs to one widget: share an instance and hiding one hides them all."""
    return widgets.Layout(width="520px")

source = widgets.Dropdown(options=["a shipped example", "a PDB ID", "a file"],
                          value="a shipped example", description="add a target from",
                          style=STYLE, layout=wide())
example = widgets.Dropdown(options=EXAMPLES, value="hPDL1" if "hPDL1" in EXAMPLES else EXAMPLES[0],
                           description="which example", style=STYLE, layout=wide())
pdb_id = widgets.Text(description="PDB ID", placeholder="4z18", style=STYLE, layout=wide())
file_path = widgets.Text(description="file", placeholder="/content/drive/MyDrive/my_target.pdb",
                         style=STYLE, layout=wide())
add = widgets.Button(description="add this target", button_style="primary",
                     layout=widgets.Layout(width="200px"))
clear = widgets.Button(description="remove all", layout=widgets.Layout(width="140px"))
rows = widgets.VBox([])
notice = widgets.Output()

# Every row is one target. The widgets are kept here rather than on the box, so nothing depends
# on being able to hang attributes off a widget.
ROWS = []

def found_target():
    """Name, file and whatever the source already knows about it."""
    if source.value == "a shipped example":
        # An example is expanded rather than named, so it can sit beside your own targets and
        # still bring its own chains, residues and role.
        preset = json.loads((REPO / "settings" / "target" / f"{example.value}.json").read_text())
        entry = dict(preset["targets"][0])
        entry["target_path"] = str((REPO / "settings" / "target" / entry["target_path"]).resolve())
        return entry
    if source.value == "a PDB ID":
        identifier = pdb_id.value.strip().lower()
        if not identifier:
            raise ValueError("fill in the PDB ID")
        path = TARGET_DIR / f"{identifier}.cif"
        if not path.exists():
            got = subprocess.run(["curl", "-fsSL", "-o", str(path),
                                  f"https://files.rcsb.org/download/{identifier}.cif"])
            if got.returncode != 0:
                path.unlink(missing_ok=True)
                raise ValueError(f"the PDB has no entry {identifier.upper()}")
        return {"name": identifier.upper(), "target_path": str(path.resolve()), "chains": "A"}
    path = Path(file_path.value.strip())
    if not path.exists():
        raise ValueError(f"no file at {path}")
    return {"name": path.stem, "target_path": str(path.resolve()), "chains": "A"}

def target_row(entry):
    """One target, editable."""
    path = Path(entry["target_path"])
    chains = widgets.Text(value=entry.get("chains", ""), description="chains",
                          placeholder="A, or A,B", style=STYLE, layout=wide())
    hotspots = widgets.Text(value=entry.get("hotspots", ""), description="residues to bind",
                            placeholder="anywhere", style=STYLE, layout=wide())
    coldspots = widgets.Text(value=entry.get("coldspots", ""),
                             description="residues to keep clear", placeholder="none",
                             style=STYLE, layout=wide())
    role = widgets.Dropdown(options=ROLES,
                            value=ROLES[1] if entry.get("objective") == "detarget" else ROLES[0],
                            description="the binder should", style=STYLE, layout=wide())
    if path.suffix.lower() in SEQUENCE_SUFFIXES:
        chains.layout.display = "none"        # a sequence target has no chains to pick
    remove = widgets.Button(description="remove", layout=widgets.Layout(width="110px"))
    heading = widgets.HTML(f"<b>{entry['name']}</b> &nbsp;<code>{path.name}</code>")
    box = widgets.VBox([widgets.HBox([heading, remove]), chains, hotspots, coldspots, role],
                       layout=widgets.Layout(border="1px solid #d0d0d0", padding="8px",
                                             margin="6px 0"))
    row = {"box": box, "name": entry["name"], "path": str(path),
           "fields": (chains, hotspots, coldspots, role)}
    remove.on_click(lambda button: drop(row))
    return row

def drop(row):
    if row in ROWS:
        ROWS.remove(row)
    redraw()

def redraw():
    rows.children = tuple(row["box"] for row in ROWS)
    with notice:
        notice.clear_output(wait=True)
        if not ROWS:
            print("No targets yet. Add at least one.")

def on_add(button=None):
    with notice:
        notice.clear_output(wait=True)
        try:
            entry = found_target()
        except ValueError as problem:
            print(problem)
            return
        if any(row["path"] == entry["target_path"] for row in ROWS):
            print(f"{entry['name']} is already here.")
            return
    ROWS.append(target_row(entry))
    redraw()

def on_clear(button=None):
    ROWS.clear()
    redraw()

def show(change=None):
    # None, not "", is what puts a hidden widget back.
    example.layout.display = None if source.value == "a shipped example" else "none"
    pdb_id.layout.display = None if source.value == "a PDB ID" else "none"
    file_path.layout.display = None if source.value == "a file" else "none"

def target_settings():
    """Read the rows now, and say how large the biggest target is."""
    entries = []
    for row in ROWS:
        chains, hotspots, coldspots, role = row["fields"]
        entry = {"name": row["name"], "target_path": row["path"]}
        for field, key in ((chains, "chains"), (hotspots, "hotspots"), (coldspots, "coldspots")):
            if field.value.strip() and field.layout.display != "none":
                entry[key] = field.value.strip()
        if role.value == ROLES[1]:
            entry["objective"] = "detarget"
        entries.append(entry)
    if not entries:
        raise ValueError("add at least one target in cell 3")
    if all(entry.get("objective") == "detarget" for entry in entries):
        raise ValueError("every target is an off-target; add one to bind")
    largest = 0
    for entry in entries:
        counts = chain_residues(Path(entry["target_path"]))
        wanted = [name.strip() for name in entry.get("chains", "").split(",") if name.strip()]
        residues = sum(counts[name] for name in wanted if name in counts) or sum(counts.values())
        largest = max(largest, residues)
    return {"targets": entries}, largest

add.on_click(on_add)
clear.on_click(on_clear)
source.observe(show, names="value")
display(widgets.VBox([source, example, pdb_id, file_path, widgets.HBox([add, clear]),
                      rows, notice]))
show()
redraw()

In [ ]:
#@title 4 · Binder { display-mode: "form" }
#@markdown Indicate binder type ans settings. Settings depend on the binder type. For example copies appear for a homo-oligomer, the lengths
#@markdown disappear for a scaffolded type.
#@markdown
#@markdown A campaign runs until it has the designs you asked for. Stop the run cell whenever you have seen enough.
try:
    import ipywidgets as widgets
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "ipywidgets"], check=True)
    import ipywidgets as widgets
from IPython.display import display

# Some Colab runtimes need this before a widget will draw.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

if "REPO" not in globals():
    raise SystemExit("Run cells 1 and 2 first.")

# The form says what a thing is; these map it to the preset and setting names BC2 uses.
BINDER_PRESETS = {"de novo binder": "binder", "large binder": "large_binder",
                  "linear peptide": "peptide", "cyclic peptide": "cyclic_peptide",
                  "homo-oligomer": "homo_oligomer", "multidomain binder": "multidomain",
                  "VHH": "VHH", "nanobody (3EAK)": "Nanobody",
                  "ankyrin repeat protein": "ARP", "scFv": "scFv", "Fab": "Fab", }
PROPERTY_SETTINGS = {"bind only the named residues": "forced_targeting",
                     "human-like sequence": "humanize",
                     "protease resistant": "protease_stable",
                     "add a disulfide staple": "disulfide_staple",
                     "more beta sheet": "mixed_topology",
                     "termini close together": "termini_together",
                     "termini point away from the target": "termini_accessible"}
SCAFFOLDED = ("VHH", "Nanobody", "ARP", "scFv", "Fab")
# Residues per scaffold, for the target-size estimate only; the scaffold sets the real length.
SCAFFOLD_LENGTHS = {"VHH": 130, "Nanobody": 130, "ARP": 130, "scFv": 250, "Fab": 460}
# A large binder is the one type with no preset range. The README says over 300.
NO_PRESET_LENGTHS = [320, 400]

def preset_lengths(name):
    """What this binder type already decides for itself."""
    return json.loads((REPO / "settings" / "modality" / f"{name}.json").read_text()).get("binder_lengths")

STYLE = {"description_width": "200px"}

def wide():
    """A Layout belongs to one widget: share an instance and hiding one hides them all."""
    return widgets.Layout(width="440px")
binder_type = widgets.Dropdown(options=list(BINDER_PRESETS), value="de novo binder",
                               description="binder type", style=STYLE, layout=wide())
shortest = widgets.BoundedIntText(value=60, min=1, max=2000, description="shortest binder",
                                  style=STYLE, layout=wide(), continuous_update=True)
longest = widgets.BoundedIntText(value=100, min=1, max=2000, description="longest binder",
                                 style=STYLE, layout=wide(), continuous_update=True)
copies = widgets.BoundedIntText(value=2, min=2, max=12, description="homo-oligomer copies",
                                style=STYLE, layout=wide())
designs_wanted = widgets.BoundedIntText(value=2, min=1, max=10000, description="designs wanted",
                                        style=STYLE, layout=wide())
PROPERTY_BOXES = {label: widgets.Checkbox(value=False, description=label, indent=False,
                                          layout=wide()) for label in PROPERTY_SETTINGS}
LENGTHS = widgets.VBox([shortest, longest])
note = widgets.Output()

def binder_choices():
    """Read the widgets now, so nothing depends on when a field last committed."""
    names = [BINDER_PRESETS[binder_type.value]]
    settings = {"number_of_final_designs": int(designs_wanted.value),
                "modality": names[0] if len(names) == 1 else names}
    preset = preset_lengths(names[0])
    if names[0] in SCAFFOLDED:
        length = SCAFFOLD_LENGTHS.get(names[0], 250)
    else:
        chosen = [int(shortest.value), int(longest.value)]
        if chosen[0] > chosen[1]:
            raise ValueError(f"the shortest binder ({chosen[0]}) is longer than the longest ({chosen[1]})")
        # Left at the preset, the preset is what applies and the campaign file stays quiet.
        if chosen != (preset or []):
            settings["binder_lengths"] = chosen
        length = chosen[1]
    if names[0] == "homo_oligomer":
        settings["copies"] = int(copies.value)
        length *= int(copies.value)
    settings.update({PROPERTY_SETTINGS[label]: True
                     for label, box in PROPERTY_BOXES.items() if box.value})
    return settings, length, names

def refresh(change=None):
    name = BINDER_PRESETS[binder_type.value]
    preset = preset_lengths(name)
    scaffolded = name in SCAFFOLDED
    # A new binder type brings its own range with it.
    if change is not None and change.get("owner") is binder_type and not scaffolded:
        shortest.value, longest.value = preset or NO_PRESET_LENGTHS
    LENGTHS.layout.display = "none" if scaffolded else None
    copies.layout.display = None if name == "homo_oligomer" else "none"
    with note:
        note.clear_output(wait=True)
        try:
            settings, _, names = binder_choices()
        except ValueError as problem:
            print(problem)
            return
        if scaffolded:
            print(f"{binder_type.value}: the scaffold sets the length")
        elif "binder_lengths" not in settings:
            print(f"length {preset[0]}-{preset[1]}, the {name} preset")
        else:
            span = settings["binder_lengths"]
            print(f"length {span[0]}-{span[1]}" + ("" if preset else ", no preset for this type"))
        if name == "homo_oligomer":
            print(f"{copies.value} copies")
        chosen = [PROPERTY_SETTINGS[label] for label, box in PROPERTY_BOXES.items() if box.value]
        print(f"properties: {', '.join(chosen) or 'none'}")
        print(f"stop at {designs_wanted.value} design(s), however many attempts that takes")
        if PROPERTY_BOXES["bind only the named residues"].value                 and not str(globals().get("residues_to_target", "")).strip()                 and "target" not in (globals().get("TARGET_SETTINGS") or {}):
            print("bind only the named residues needs residues to target in cell 3")

for control in (binder_type, shortest, longest, copies, designs_wanted,
                *PROPERTY_BOXES.values()):
    control.observe(refresh, names="value")

display(widgets.VBox([binder_type, LENGTHS, copies, designs_wanted,
                      widgets.HTML("<b>properties</b>"), *PROPERTY_BOXES.values(), note]))
refresh()

---

## Run

Leave the cell running. Each accepted design is written as soon as it is found, and running the
cell again carries on from where it stopped.

In [ ]:
#@title 5 · Run the campaign { display-mode: "form" }
#@markdown The log names each attempt, the stage it reached and any filter a candidate failed.
#@markdown
#@markdown `no_filters` is for testing the notebook, not for designing: it opens every stage gate
#@markdown and switches every filter off, so the first attempt is accepted and the cells below
#@markdown have something to show. Nothing it accepts means anything.
no_filters = True  #@param {type:"boolean"}

import csv, selectors
try:
    import ipywidgets as widgets
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "ipywidgets"], check=True)
    import ipywidgets as widgets
from IPython.display import display

# A stage takes minutes and prints nothing while it runs, so a quiet stretch this long gets a
# line saying where the campaign has got to.
HEARTBEAT = 60

started = time.time()

def rows_in(path):
    """How many rows a table holds, without reading it all."""
    try:
        with open(path) as table:
            return max(0, sum(1 for _ in table) - 1)
    except OSError:
        return 0

def rejected_in(path):
    """Candidates that were scored and did not pass."""
    try:
        with open(path, newline="") as table:
            return sum(1 for row in csv.DictReader(table)
                       if (row.get("outcome") or "").strip() not in ("", "passed"))
    except OSError:
        return 0

def elapsed():
    minutes, seconds = divmod(int(time.time() - started), 60)
    return f"{minutes}m{seconds:02d}s"

def save_campaign():
    """Write the campaign the cells above describe, and say what it asks of the card. A campaign
    picked in cell 2 already has its file, and is left alone."""
    global PROJECT_FOLDER, CAMPAIGN_FILE, CAMPAIGN_SETTINGS
    if globals().get("REUSED_CAMPAIGN"):
        if CAMPAIGN_FILE.exists():
            return CAMPAIGN_FILE
        raise SystemExit(f"{CAMPAIGN_FILE} is not there, so there is nothing to run.")

    # One design at a time on one card, and no fan-out: a campaign that decides to spread itself
    # re-launches as worker processes, and their log would not be the run cell's. Loss plots cost
    # time nothing here reads; the animations are what the download carries.
    LENGTH_BUCKET = 32
    RUN_SETTINGS = {"save_loss_plots": False,
                    "save_design_animations": True,
                    "length_bucket_size": LENGTH_BUCKET,
                    "workers_per_gpu": 1,
                    "auto_multi_gpu": False}

    # Read both sets of widgets now rather than trusting what they last left behind.
    try:
        TARGET_SETTINGS, TARGET_RESIDUES = target_settings()
        BINDER_SETTINGS, BINDER_LENGTH, MODALITY_NAMES = binder_choices()
    except ValueError as problem:
        raise SystemExit(str(problem))

    target_label = "_".join(entry["name"] for entry in TARGET_SETTINGS["targets"]
                            if entry.get("objective") != "detarget")
    name = new_name.value.strip() or "_".join([str(target_label), *MODALITY_NAMES])
    PROJECT_FOLDER = RESULTS_ROOT / name
    PROJECT_FOLDER.mkdir(parents=True, exist_ok=True)
    CAMPAIGN_FILE = PROJECT_FOLDER / "campaign.json"

    CAMPAIGN_SETTINGS = {**TARGET_SETTINGS, **BINDER_SETTINGS, **RUN_SETTINGS,
                         "project_folder": str(PROJECT_FOLDER)}
    CAMPAIGN_FILE.write_text(json.dumps(CAMPAIGN_SETTINGS, indent=2) + "\n")
    chosen_name = new_name.value.strip() or "_".join([str(target_label), *MODALITY_NAMES])
    PROJECT_FOLDER = RESULTS_ROOT / chosen_name
    PROJECT_FOLDER.mkdir(parents=True, exist_ok=True)
    CAMPAIGN_FILE = PROJECT_FOLDER / "campaign.json"
    CAMPAIGN_SETTINGS = {**TARGET_SETTINGS, **BINDER_SETTINGS, **RUN_SETTINGS,
                         "project_folder": str(PROJECT_FOLDER)}
    CAMPAIGN_FILE.write_text(json.dumps(CAMPAIGN_SETTINGS, indent=2) + "\n")
    print(CAMPAIGN_FILE.read_text())

    # One design's budget, from the formula the campaign plans with: 2.0 x (3.4 GB + 38 kB x N^2)
    # for a padded complex of N residues, with 4 GB of the card left over.
    padded = -(-(TARGET_RESIDUES + BINDER_LENGTH) // LENGTH_BUCKET) * LENGTH_BUCKET
    per_worker = 2.0 * (3.4 + 38e-6 * padded ** 2)
    usable = GPU_MEMORY_GB - 4
    print(f"{TARGET_RESIDUES} + {BINDER_LENGTH} residues, padded to {padded}")
    print(f"needs about {per_worker:.0f} GB" + (f", {usable:.0f} GB usable" if GPU_MEMORY_GB else ""))
    if not GPU_MEMORY_GB:
        print("This card did not report its memory, so nothing here can say whether it fits.")
    elif TARGET_RESIDUES == 0:
        print("The target residues could not be counted, so this covers the binder only.")
    elif per_worker > usable:
        print("Too big for this card. Trim the target or shorten the binder.")
    elif per_worker > usable * 0.8:
        print("Close to the limit. Trim the target if it fails.")
    else:
        print("Fits.")
    return CAMPAIGN_FILE

CAMPAIGN_FILE = save_campaign()

counts = widgets.HTML()
log = widgets.Output(layout=widgets.Layout(max_height="440px", overflow="auto",
                                           border="1px solid #e0e0e0", padding="4px"))
display(widgets.VBox([counts, log]))

def tally():
    folder = globals().get("PROJECT_FOLDER") or Path(".")
    passing = rows_in(folder / "3_Ranked" / "!_Ranked.csv")
    failed = rejected_in(folder / "2_Refolded" / "!_Refolded.csv")
    attempts = rows_in(folder / "1_Trajectories" / "!_Trajectories.csv")
    counts.value = (
        "<div style='font-family:monospace;font-size:15px;padding:4px 0'>"
        f"<b style='color:#137333'>{passing} passing</b>"
        f"&nbsp;&nbsp;<b style='color:#c5221f'>{failed} failed</b>"
        f"&nbsp;&nbsp;<span style='color:#5f6368'>{attempts} attempts · {elapsed()}</span></div>")
    return passing

def without_filters(campaign_file):
    """Open every stage gate and switch every filter off, reading the names out of the presets
    this campaign uses, and write that beside it rather than over it."""
    settings = json.loads(Path(campaign_file).read_text())
    named = settings.get("modality") or "binder"
    named = [named] if isinstance(named, str) else list(named)
    presets = [REPO / "settings" / "core" / "default.json"]
    presets += [REPO / "settings" / "modality" / f"{name}.json" for name in named]
    presets += [REPO / "settings" / "property" / f"{name}.json"
                for name, value in settings.items() if value is True]
    for path in presets:
        if not path.exists():
            continue
        preset = json.loads(path.read_text())
        for key in preset:
            # only the per-stage gates, which is what leaves the campaign's counters alone
            if key.startswith(("min_", "max_")) and re.search(
                    r"_(screen|refine|anneal|harden|mutate|final)$", key):
                settings[key] = 0 if key.startswith("min_") else 1000000
        settings.setdefault("filters", {})
        settings["filters"].update({metric: {"threshold": None}
                                    for metric in preset.get("filters", {})})
    relaxed = Path(campaign_file).with_name("campaign_no_filters.json")
    relaxed.write_text(json.dumps(settings, indent=2) + "\n")
    return relaxed

RUNNING = without_filters(CAMPAIGN_FILE) if no_filters else CAMPAIGN_FILE

tally()
with log:
    if no_filters:
        print("no_filters: every gate open and every filter off. For testing only.")
    print(f"{RUNNING}\n")

# PYTHONUNBUFFERED because a pipe makes Python hold output back, and what is held back looks
# exactly like a campaign that has stopped.
campaign = subprocess.Popen([*LAUNCHER, str(RUNNING)], cwd=REPO,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
watcher = selectors.DefaultSelector()
watcher.register(campaign.stdout, selectors.EVENT_READ)
spoke_at = time.time()
status = "interrupted"
try:
    while True:
        ready = watcher.select(timeout=max(1, min(5, HEARTBEAT)))
        tally()
        if ready:
            line = campaign.stdout.readline()
            if line == "":
                break
            with log:
                print(line, end="")
            spoke_at = time.time()
            continue
        quiet = time.time() - spoke_at
        if quiet >= HEARTBEAT:
            with log:
                print(f"   [{elapsed()}] still working, quiet for {int(quiet)}s", flush=True)
            spoke_at = time.time()
        if campaign.poll() is not None and not watcher.select(timeout=0):
            break
    status = campaign.wait()
except KeyboardInterrupt:
    campaign.terminate()
    campaign.wait()
    with log:
        print("\nStopped. Accepted designs are kept. Run this cell again to carry on.")
finally:
    watcher.close()

accepted = tally()
with log:
    print(f"\n{'-' * 60}\nexit {status} after {elapsed()}, {accepted} design(s) accepted")
    if status == 2:
        print("Exit 2 is a refusal. Every reason is above. Fix cells 3-4, re-run cell 5, then here.")
    elif status == 0:
        print("Done. Cell 7 shows what was accepted.")

---

# Results

These read whichever campaign cell 2 loaded, or the one you just ran.

In [ ]:
#@title 6 · Results { display-mode: "form" }
#@markdown Accepted designs, best first by `i_pDAE` — interface confidence, 0 to 1, higher is
#@markdown better. If nothing was accepted, this shows the candidates and the filters that
#@markdown rejected them.
show_rows = 20  #@param {type:"integer"}

import pandas as pd
pd.set_option("display.max_colwidth", 60)

# The campaign cell 7 loaded, else the one cell 6 wrote.
CAMPAIGN_FOLDER = campaign_folder() if "campaign_folder" in globals() else globals().get("PROJECT_FOLDER")
if CAMPAIGN_FOLDER is None or not Path(CAMPAIGN_FOLDER).exists():
    raise SystemExit("Write a campaign in cell 6, or load one in cell 7.")
CAMPAIGN_FOLDER = Path(CAMPAIGN_FOLDER)

ACCEPTED_COLUMNS = ["rank", "design", "length", "i_pDAE", "i_pTM", "pLDDT", "i_pAE",
                    "Interface_Residues", "Interface_BuriedArea", "Surface_Hydrophobicity",
                    "Binder_Mass_kDa", "Binder_pI", "Binder_Sequence"]
CANDIDATE_COLUMNS = ["design", "outcome", "failed_filters", "i_pDAE", "i_pTM", "pLDDT", "i_pAE",
                     "Interface_Residues", "Binder_Sequence"]
ATTEMPT_COLUMNS = ["design", "trajectory", "terminated", "i_pTM", "pLDDT"]

def stage_table(*names):
    for relative in names:
        path = CAMPAIGN_FOLDER / relative
        if path.exists():
            return pd.read_csv(path), path
    return None, None

ACCEPTED, accepted_path = stage_table("3_Ranked/!_Ranked.csv", "ranked.csv", "accepted.csv")
CANDIDATES, _ = stage_table("2_Refolded/!_Refolded.csv", "candidates.csv")
ATTEMPTS, _ = stage_table("1_Trajectories/!_Trajectories.csv", "trajectories.csv")

def show(frame, columns):
    display(frame[[column for column in columns if column in frame.columns]].head(show_rows))

for label, frame in (("attempts", ATTEMPTS), ("candidates", CANDIDATES), ("accepted", ACCEPTED)):
    print(f"{0 if frame is None else len(frame):>5} {label}")

if ACCEPTED is not None and len(ACCEPTED):
    show(ACCEPTED, ACCEPTED_COLUMNS)
elif CANDIDATES is not None and len(CANDIDATES):
    print("\nNothing accepted yet. Candidates, and the filters they failed:")
    show(CANDIDATES, CANDIDATE_COLUMNS)
    if "failed_filters" in CANDIDATES:
        counted = CANDIDATES["failed_filters"].dropna().str.split(",").explode().str.strip().value_counts()
        print("\nrejected most often by:")
        print(counted.head(10).to_string())
elif ATTEMPTS is not None and len(ATTEMPTS):
    print("\nNo candidate reached scoring. `terminated` is where each attempt stopped:")
    show(ATTEMPTS, ATTEMPT_COLUMNS)
else:
    print("\nNothing written yet.")

In [ ]:
#@title 7 · Structures { display-mode: "form" }
#@markdown The accepted designs, best first. Target chains are slate, the binder is rose.
#@markdown pLDDT colours by confidence instead: blue is confident, red is not.
how_many = 4  #@param {type:"integer"}
colour_by = "chain"  #@param ["chain", "pLDDT"]

# The campaign cell 7 loaded, else the one cell 6 wrote.
CAMPAIGN_FOLDER = campaign_folder() if "campaign_folder" in globals() else globals().get("PROJECT_FOLDER")
if CAMPAIGN_FOLDER is None or not Path(CAMPAIGN_FOLDER).exists():
    raise SystemExit("Write a campaign in cell 6, or load one in cell 7.")
CAMPAIGN_FOLDER = Path(CAMPAIGN_FOLDER)

try:
    import py3Dmol
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "py3Dmol"], check=True)
    import py3Dmol

folder = CAMPAIGN_FOLDER / "3_Ranked" if (CAMPAIGN_FOLDER / "3_Ranked").exists() else CAMPAIGN_FOLDER / "accepted"
structures = [path for path in sorted(folder.glob("*.cif")) if "_monomer" not in path.name]              if folder.exists() else []
if not structures:
    raise SystemExit(f"No accepted structures in {folder}. Cell 7 shows how far the campaign got.")

# The table's order is the one to trust; filenames are not sorted by rank.
order = list(ACCEPTED["design"]) if ACCEPTED is not None and "design" in ACCEPTED else []
by_rank = [path for name in order for path in structures if path.stem.startswith(str(name))] or structures
shown = by_rank[:max(1, how_many)]

columns = min(2, len(shown))
rows = -(-len(shown) // columns)
view = py3Dmol.view(viewergrid=(rows, columns), width=460 * columns, height=360 * rows)
palette = ["#3c5b6f", "#B76E79", "#8FA98F", "#C9A227"]
for index, path in enumerate(shown):
    where = (index // columns, index % columns)
    view.addModel(path.read_text(), "cif", viewer=where)
    if colour_by == "pLDDT":
        view.setStyle({}, {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb",
                                                       "min": 50, "max": 90}}}, viewer=where)
    else:
        for number, chain in enumerate(sorted(chain_residues(path)) or ["A", "B"]):
            view.setStyle({"chain": chain},
                          {"cartoon": {"color": palette[number % len(palette)]}}, viewer=where)
    view.zoomTo(viewer=where)
view.setBackgroundColor("white")

for index, path in enumerate(shown, start=1):
    score = ""
    if ACCEPTED is not None and "i_pDAE" in ACCEPTED.columns:
        row = ACCEPTED[ACCEPTED["design"].astype(str).apply(lambda name: path.stem.startswith(name))]
        if len(row):
            score = f"  i_pDAE {row.iloc[0]['i_pDAE']}"
    print(f"{index}. {path.name}{score}")
view.show()

In [ ]:
#@title 8 · Download { display-mode: "form" }
#@markdown A zip of the accepted designs: their structures, their animations, the ranked table,
#@markdown the sequences as FASTA, and the campaign metadata that records the settings behind
#@markdown them.
#@markdown
#@markdown If the download does not start, disable your ad blocker, or take the zip from the
#@markdown folder icon on the left.
how_many_sequences = 10  #@param {type:"integer"}

# The campaign cell 2 loaded, else the one cell 5 wrote.
CAMPAIGN_FOLDER = campaign_folder() if "campaign_folder" in globals() else globals().get("PROJECT_FOLDER")
if CAMPAIGN_FOLDER is None or not Path(CAMPAIGN_FOLDER).exists():
    raise SystemExit("Write a campaign in cell 5, or load one in cell 2.")
CAMPAIGN_FOLDER = Path(CAMPAIGN_FOLDER)

frame = ACCEPTED
if frame is None or not len(frame):
    raise SystemExit("Nothing accepted yet. Run cell 7 first.")
if "Binder_Sequence" not in frame.columns:
    raise SystemExit("No Binder_Sequence column to export.")

chosen = frame.head(how_many_sequences)
records = []
for _, row in chosen.iterrows():
    design = str(row.get("design", "design"))
    # BC2 writes a multi-chain binder in one cell, separated by /, which is not a sequence.
    chains_out = [chain for chain in str(row["Binder_Sequence"]).split("/") if chain]
    for number, sequence in enumerate(chains_out, start=1):
        label = design if len(chains_out) == 1 else f"{design}_chain{number}"
        notes = " ".join(f"{key}={row[key]}" for key in ("rank", "i_pDAE", "i_pTM") if key in frame.columns)
        records.append(f">{label} {notes}\n{sequence}")

export = RUNTIME / "export" / CAMPAIGN_FOLDER.name
shutil.rmtree(export.parent, ignore_errors=True)
export.mkdir(parents=True)
(export / f"{CAMPAIGN_FOLDER.name}_binders.fasta").write_text("\n".join(records) + "\n")

# The finished designs and their animations, both of which BC2 writes into 3_Ranked, plus what
# says where they came from.
ranked = CAMPAIGN_FOLDER / "3_Ranked"
designs = sorted(ranked.glob("*.cif")) if ranked.exists() else []
animations = sorted(ranked.glob("*.html")) if ranked.exists() else []
for source_file in designs + animations:
    shutil.copy2(source_file, export / source_file.name)
for beside in ("3_Ranked/!_Ranked.csv", "campaign_metadata.json", "campaign.json"):
    if (CAMPAIGN_FOLDER / beside).exists():
        shutil.copy2(CAMPAIGN_FOLDER / beside, export / Path(beside).name)

archive = shutil.make_archive(str(RUNTIME / f"{CAMPAIGN_FOLDER.name}_designs"), "zip",
                              export.parent, export.name)
print(f"{len(designs)} design(s), {len(animations)} animation(s), {len(records)} sequence(s)")
print(f"{archive} ({Path(archive).stat().st_size / 1e6:.0f} MB)")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not in Colab: the zip is at the path above.")

---

## Output Format

| File | What it is |
| --- | --- |
| `3_Ranked/!_Ranked.csv` | Accepted designs, best first by `i_pDAE`. Start here. |
| `3_Ranked/*.cif` | The predicted complexes. B-factors hold pLDDT, 0–100. |
| `2_Refolded/!_Refolded.csv` | Every candidate, with the filters it failed. |
| `1_Trajectories/` | One folder per attempt, with its plots. |
| `campaign_metadata.json` | Settings, model versions, checkpoint hashes. Keep it. |

`i_pDAE` and `i_pTM` are interface confidence, 0 to 1, higher is better. `i_pAE` is interface
error, lower is better. `pLDDT` is binder confidence. `Interface_Residues` is interface size.
High `Surface_Hydrophobicity` may aggregate. The rest are in
[the output reference](https://github.com/PacesaLab/BindCraft2/blob/main/docs/outputs.md).

